# Log Extractor — SSL / RoboCup

Notebook focado **apenas em extrair os logs do robô (`.log.gz`) e salvar em formato legível** (CSV + Parquet).

Toda a parte de detecção de chute, segmentação por evento e plots foi removida — esse notebook é só o pipeline ETL.

**Saídas geradas** em `OUTPUT_DIR/<log_name>/`:
- `ball.csv` / `.parquet` — bola processada (visão)
- `raw_ball.csv` / `.parquet` — bola crua (SSL-Vision)
- `processed_robots.csv` / `.parquet` — robôs processados
- `raw_robots.csv` / `.parquet` — robôs crus (SSL-Vision)
- `commands.csv` / `.parquet` — comandos enviados aos aliados
- `telemetry.csv` / `.parquet` — telemetria dos aliados
- `referee.csv` / `.parquet` — game states
- `manifest.json` — metadados da extração (contagem, time-ref, duração)

## 1. Requisitos

In [4]:
pip install -q pandas numpy "betterproto[compiler]" pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


## 2. Geração das classes Protobuf

In [5]:
from utils.auto_generate_proto import generate_proto_classes
generate_proto_classes()

CMD: C:\ProgramData\chocolatey\bin\protoc.EXE --plugin=protoc-gen-python_betterproto=C:\Users\victo\Documents\digitaltwin\scripts\local\.protoc_plugins\protoc-gen-python_betterproto.cmd -I C:\Users\victo\Documents\in1144-data-mining-2025-2\libs\protobufs\include\protobufs\pb\proto --python_betterproto_out=C:\Users\victo\Documents\in1144-data-mining-2025-2\proto\generated rc_log.proto
returncode: 0
STDOUT:
 
STDERR:
 Writing RCLog.py
Writing RoboCupSSL.py
Writing __init__.py

✅ Sucesso!


## 3. Imports e configuração

In [8]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

from data_select.data_filter_operator import *
from utils.utils import LogFields, gel_2d_length_in_column

# ------------------ CONFIG ------------------
PREFIX_PATH = 'logs/'
LOG_FILE    = '2026-05-18_19-2-15.log.gz'   # <-- troque pelo log atual
OUTPUT_DIR  = 'extracted'

SAVE_CSV     = True
SAVE_PARQUET = True   # parquet é ~10x menor e bem mais rápido pra recarregar
# --------------------------------------------

log_path  = Path(PREFIX_PATH) / LOG_FILE
out_dir   = Path(OUTPUT_DIR) / Path(LOG_FILE).stem.replace('.log', '')
out_dir.mkdir(parents=True, exist_ok=True)
print(f'Log:    {log_path}')
print(f'Saída:  {out_dir.resolve()}')

Log:    logs\2026-05-18_19-2-15.log.gz
Saída:  C:\Users\victo\Documents\digitaltwin\scripts\local\extracted\2026-05-18_19-2-15


## 4. Carregamento do log

In [12]:
selects = [
    LogFields.RAW_FRAME,
    LogFields.REFEREE,
    LogFields.TELEMETRY,
    LogFields.ROBOTS_COMMAND,
    LogFields.PROCESSED_FRAME,
    # adicione PROCESSED_FRAME aqui caso não venha junto com RAW_FRAME no seu loader
]

data_list = load_select_modules(str(log_path), selects)
print(f'Pacotes carregados: {len(data_list)}')

Pacotes carregados: 59310


## 5. Funções extratoras

Cada função recebe a lista `data_list` (dicionários decodificados do protobuf) e devolve um `DataFrame` plano.

In [13]:
def extract_referee(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        ref = d.get('referee')
        if ref is None:
            continue
        refs = ref if isinstance(ref, list) else [ref]
        for r in refs:
            if not isinstance(r, dict):
                continue
            ts = r.get('publish_timestamp')
            gs = r.get('game_state_name') or (r.get('game_state', {}) or {}).get('name')
            if ts is None or gs is None:
                continue
            rows.append({'timestamp': float(ts), 'game_state_name': str(gs)})
    return pd.DataFrame(rows)


def extract_processed_robots(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        pf = d.get('processed_frame')
        if not pf:
            continue
        ts = pf.get('publish_timestamp', 0)
        if not ts:
            continue
        for team in ('allies', 'enemies'):
            for robot in pf.get(team, []) or []:
                pos = robot.get('position', {}) or {}
                vel = robot.get('velocity', {}) or {}
                rows.append({
                    'timestamp': ts,
                    'team': team,
                    'robot_id': robot.get('id'),
                    'position_x': pos.get('x'),
                    'position_y': pos.get('y'),
                    'position_w': pos.get('omega'),
                    'velocity_x': vel.get('x'),
                    'velocity_y': vel.get('y'),
                })
    return pd.DataFrame(rows)


def extract_raw_robots(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        for rf in d.get('raw_frame', []) or []:
            ts = rf.get('publish_timestamp', 0)
            if not ts:
                continue
            detection = (rf.get('packet', {}) or {}).get('detection', {}) or {}
            for team in ('robots_blue', 'robots_yellow'):
                for robot in detection.get(team, []) or []:
                    rows.append({
                        'timestamp': ts,
                        'team': team,
                        'robot_id': robot.get('robot_id'),
                        'position_x': robot.get('x'),
                        'position_y': robot.get('y'),
                        'position_w': robot.get('orientation'),
                    })
    return pd.DataFrame(rows)


def extract_processed_ball(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        pf = d.get('processed_frame')
        if not pf:
            continue
        ball = pf.get('ball')
        ts = pf.get('publish_timestamp', 0)
        if not ball or not ts:
            continue
        pos = ball.get('position', {}) or {}
        vel = ball.get('velocity', {}) or {}
        acc = ball.get('acceleration', {}) or {}
        rows.append({
            'timestamp': ts,
            'position_x': pos.get('x'), 'position_y': pos.get('y'),
            'velocity_x': vel.get('x'), 'velocity_y': vel.get('y'),
            'acceleration_x': acc.get('x'), 'acceleration_y': acc.get('y'),
        })
    return pd.DataFrame(rows)


def extract_raw_ball(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        for rf in d.get('raw_frame', []) or []:
            ts = rf.get('publish_timestamp', 0)
            if not ts:
                continue
            balls = ((rf.get('packet', {}) or {}).get('detection', {}) or {}).get('balls', []) or []
            for b in balls:
                rows.append({
                    'timestamp': ts,
                    'position_x': b.get('x'),
                    'position_y': b.get('y'),
                })
    return pd.DataFrame(rows)


def extract_commands(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        rc = d.get('robots_command')
        if not rc:
            continue
        for robot in rc.get('navigation', []) or []:
            ts = robot.get('publish_timestamp', 0)
            if not ts:
                continue
            mv = robot.get('move', {}) or {}
            ac = robot.get('actuation', {}) or {}
            rows.append({
                'timestamp': ts,
                'robot_id': robot.get('id'),
                'move_x': mv.get('x'), 'move_y': mv.get('y'), 'move_w': mv.get('omega'),
                'actuation_kick_strength':     ac.get('kick_strength'),
                'actuation_front':             ac.get('front'),
                'actuation_chip':              ac.get('chip'),
                'actuation_charge':            ac.get('charge'),
                'actuation_dribbler':          ac.get('dribbler'),
                'actuation_dribbler_velocity': ac.get('dribbler_velocity'),
            })
    return pd.DataFrame(rows)


def extract_telemetry(data_list: list) -> pd.DataFrame:
    rows = []
    for d in data_list:
        tl = d.get('telemetry')
        if not tl:
            continue
        for robot in tl.get('telemetries', []) or []:
            ts = robot.get('receive_timestamp', 0)
            if not ts:
                continue
            pos = robot.get('position', {}) or {}
            vel = robot.get('velocity', {}) or {}
            row = {
                'timestamp': ts,
                'robot_id': robot.get('id'),
                'position_x': pos.get('x'), 'position_y': pos.get('y'), 'position_w': pos.get('omega'),
                'velocity_x': vel.get('x'), 'velocity_y': vel.get('y'), 'velocity_w': vel.get('omega'),
                'dribbler_speed':        robot.get('dribbler_speed'),
                'capacitor_charge':      robot.get('capacitor_charge'),
                'dribbler_ball_contact': robot.get('dribbler_ball_contact'),
                'battery':               robot.get('battery'),
                'count':                 robot.get('count'),
            }
            for i in range(1, 5):
                row[f'wheel{i}'] = robot.get(f'wheel{i}')
            rows.append(row)
    return pd.DataFrame(rows)

## 6. Execução da extração

In [14]:
dfs = {
    'ball':             extract_processed_ball(data_list),
    'raw_ball':         extract_raw_ball(data_list),
    'processed_robots': extract_processed_robots(data_list),
    'raw_robots':       extract_raw_robots(data_list),
    'commands':         extract_commands(data_list),
    'telemetry':        extract_telemetry(data_list),
    'referee':          extract_referee(data_list),
}

for name, df in dfs.items():
    print(f'{name:>20s}: {len(df):>8d} linhas, {len(df.columns):>2d} colunas')

                ball:    59310 linhas,  7 colunas
            raw_ball:    14150 linhas,  3 colunas
    processed_robots:    27408 linhas,  8 colunas
          raw_robots:    16131 linhas,  6 colunas
            commands:    48012 linhas, 11 colunas
           telemetry:    57415 linhas, 17 colunas
             referee:    27407 linhas,  2 colunas


## 7. Normalização de timestamps

Os timestamps vêm em **nanossegundos absolutos**. Aqui:
1. convertemos para `float`,
2. descartamos linhas com `timestamp <= 1` (lixo),
3. tomamos como referência o menor timestamp da bola (processed ou raw),
4. devolvemos um eixo de tempo em **segundos a partir de t=0**.

Também adicionamos colunas de norma (`velocity_norm`, `acceleration_norm`) onde fazem sentido.

In [15]:
# 1) cast + filtro
for name, df in dfs.items():
    if df.empty:
        continue
    df['timestamp'] = df['timestamp'].astype(float)
    dfs[name] = df[df['timestamp'] > 1].reset_index(drop=True)

# 2) referência temporal (menor ts entre ball e raw_ball)
candidates = [d['timestamp'].iloc[0] for d in (dfs['ball'], dfs['raw_ball']) if not d.empty]
if not candidates:
    raise RuntimeError('Nem ball nem raw_ball têm dados — confira o log.')
timeref = min(candidates)
print(f'timeref (ns): {timeref:.0f}')

# 3) ns -> s a partir de t=0
def _to_seconds(ts: pd.Series) -> pd.Series:
    # o %1e13 preserva a lógica original (lida com possível overflow de epoch grande)
    return ((ts - timeref) % 1e13) / 1e9

for name, df in dfs.items():
    if df.empty:
        continue
    df['timestamp'] = _to_seconds(df['timestamp'])
    dfs[name] = df.sort_values('timestamp').reset_index(drop=True)

# 4) colunas derivadas
if not dfs['ball'].empty:
    dfs['ball']['velocity_norm']     = gel_2d_length_in_column(dfs['ball'], 'velocity')
    dfs['ball']['acceleration_norm'] = gel_2d_length_in_column(dfs['ball'], 'acceleration')
if not dfs['processed_robots'].empty:
    dfs['processed_robots']['velocity_norm'] = gel_2d_length_in_column(dfs['processed_robots'], 'velocity')

# checagem rápida
for name, df in dfs.items():
    if df.empty:
        continue
    print(f'{name:>20s}: t=[{df["timestamp"].iloc[0]:.3f}, {df["timestamp"].iloc[-1]:.3f}]s')

timeref (ns): 1779144202349098496
                ball: t=[2.041, 1356.748]s
            raw_ball: t=[0.000, 1355.208]s
    processed_robots: t=[2.079, 1356.748]s
          raw_robots: t=[0.000, 1356.745]s
            commands: t=[2.075, 1356.750]s
           telemetry: t=[28.240, 1356.764]s
             referee: t=[2.042, 1356.748]s


## 8. Salvamento

Cada DataFrame vira um par `<nome>.csv` + `<nome>.parquet`, mais um `manifest.json` com metadados.

In [16]:
manifest = {
    'log_file':   str(log_path),
    'timeref_ns': float(timeref),
    'tables':     {},
}

for name, df in dfs.items():
    base = out_dir / name
    if SAVE_CSV:
        df.to_csv(base.with_suffix('.csv'), index=False)
    if SAVE_PARQUET and not df.empty:
        df.to_parquet(base.with_suffix('.parquet'), index=False)
    manifest['tables'][name] = {
        'rows':     int(len(df)),
        'columns':  list(df.columns),
        't_start':  float(df['timestamp'].iloc[0])  if not df.empty else None,
        't_end':    float(df['timestamp'].iloc[-1]) if not df.empty else None,
    }

with open(out_dir / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'✅ Extração salva em: {out_dir.resolve()}')
for p in sorted(out_dir.iterdir()):
    print(f'  - {p.name:25s} ({p.stat().st_size/1024:8.1f} KB)')

✅ Extração salva em: C:\Users\victo\Documents\digitaltwin\scripts\local\extracted\2026-05-18_19-2-15
  - ball.csv                  (  4424.1 KB)
  - ball.parquet              (  2264.7 KB)
  - commands.csv              (  4957.1 KB)
  - commands.parquet          (  1697.2 KB)
  - manifest.json             (     2.4 KB)
  - processed_robots.csv      (  3686.0 KB)
  - processed_robots.parquet  (  1856.9 KB)
  - raw_ball.csv              (   703.3 KB)
  - raw_ball.parquet          (   334.1 KB)
  - raw_robots.csv            (  1369.1 KB)
  - raw_robots.parquet        (   513.3 KB)
  - referee.csv               (   668.4 KB)
  - referee.parquet           (   266.4 KB)
  - telemetry.csv             (  6000.7 KB)
  - telemetry.parquet         (   555.1 KB)


## 9. (Opcional) Reler depois

```python
import pandas as pd
ball = pd.read_parquet('extracted/<log_name>/ball.parquet')
```